In [1]:
#!/usr/bin/env python3
import os
import numpy as np
import nibabel as nib

# ==========================================
# 1. 路径配置 (请替换为包含未压缩数据的真实路径)
# ==========================================
base_dir = "/home/liujia/g_linux/Phantom_Carotid_Muscle_33"
base_dir = "/home/liujia/g_linux/test/Simu_3_33_75/standard"
# 假设这些是你还没做 log 压缩的原始文件夹
folders = [
    "Recon_HQ_33", 
    "Recon_LQ_03", 
    "Recon_SQ_75"
]

# 动态范围设置
dynamic_range = -60.0  # 超声标准：-60 dB

print("======================================================")
print(f"🔊 启动 NIfTI 数据 Log 压缩引擎 (目标范围: {dynamic_range} dB ~ 0 dB)")
print("======================================================")

for folder_name in folders:
    src_dir = os.path.join(base_dir, folder_name)
    # 自动新建带 _dB 后缀的文件夹用于保存结果
    dst_dir = os.path.join(base_dir, folder_name + "_dB")
    
    if not os.path.exists(src_dir):
        print(f"\n⚠️ 找不到源文件夹: {src_dir}，跳过。")
        continue
        
    os.makedirs(dst_dir, exist_ok=True)
    
    print(f"\n📂 正在压缩文件夹: {folder_name}")
    print("-" * 50)
    
    count = 0
    for filename in sorted(os.listdir(src_dir)):
        if filename.endswith(".nii") or filename.endswith(".nii.gz"):
            src_file = os.path.join(src_dir, filename)
            dst_file = os.path.join(dst_dir, filename)
            
            # 如果已经压缩过，跳过 (支持断点续传)
            if os.path.exists(dst_file):
                continue
                
            try:
                # 1. 读取未压缩的线性数据
                img = nib.load(src_file)
                data_linear = img.get_fdata()
                
                # 2. 取绝对值并寻找最大值 (防止复数或异常负数)
                data_abs = np.abs(data_linear)
                max_val = np.max(data_abs)
                if max_val == 0:
                    max_val = 1.0  # 防止全黑图像除以 0
                
                # 3. 核心数学：Log 压缩
                # 加 1e-12 是为了防止 log10(0) 报错引发警告
                data_db = 20 * np.log10((data_abs / max_val) + 1e-12)
                
                # 4. 动态范围截断 (Clipping)
                data_db = np.clip(data_db, dynamic_range, 0.0)
                
                # 强制转换为单精度浮点 (float32)，减小文件体积
                data_db = data_db.astype(np.float32)
                
                # 5. 重新打包并保存
                # 完好保留原来的仿射矩阵和 Header (包括 spacing)
                new_img = nib.Nifti1Image(data_db, img.affine, img.header)
                new_img.set_data_dtype(np.float32)
                nib.save(new_img, dst_file)
                
                print(f"  ✅ 压缩完成: {filename} (Min: {np.min(data_db):.2f}, Max: {np.max(data_db):.2f})")
                count += 1
                
            except Exception as e:
                print(f"  ❌ 处理失败 ({filename}): {e}")
                
    print(f"🎉 {folder_name} 处理完毕！共压缩了 {count} 个文件。")

print("\n======================================================")
print("🏆 所有未压缩数据均已成功转换为对数域 (Log Compressed)！")

🔊 启动 NIfTI 数据 Log 压缩引擎 (目标范围: -60.0 dB ~ 0 dB)

📂 正在压缩文件夹: Recon_HQ_33
--------------------------------------------------
  ✅ 压缩完成: SimData_NII_001_Pts_Unknown_hq_33ang.nii (Min: -60.00, Max: 0.00)
🎉 Recon_HQ_33 处理完毕！共压缩了 1 个文件。

📂 正在压缩文件夹: Recon_LQ_03
--------------------------------------------------
  ✅ 压缩完成: SimData_NII_0000_Pts_Unknown_lq_3ang.nii (Min: -60.00, Max: 0.00)
🎉 Recon_LQ_03 处理完毕！共压缩了 1 个文件。

📂 正在压缩文件夹: Recon_SQ_75
--------------------------------------------------
  ✅ 压缩完成: SimData_NII_001_Pts_Unknown_sq_75ang.nii (Min: -60.00, Max: 0.00)
🎉 Recon_SQ_75 处理完毕！共压缩了 1 个文件。

🏆 所有未压缩数据均已成功转换为对数域 (Log Compressed)！
